[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesamKhallaf73/pytorch-ground-up-notebooks/blob/main/notebooks/L17.ipynb)

# Lecture 17 — Moving to GPU & Fighting Overfitting

*Run the same loop on any machine — and make what it learns hold up on data it has never seen.*

**PyTorch from Ground Up · Volume I · Chapters 30–31** — companion notebook.

**How to use it:** run the cells from top to bottom. Before each code cell, *predict* what it will print — then run it and compare with the expected output (click ▸ to open it). Every exercise from the lecture is at the end.

By the end you will be able to:
- Detect the available device once and write device-agnostic code.
- Move the model and each batch correctly, and fix the device-mismatch error.
- Bring results back to the CPU and load checkpoints across machines with `map_location`.
- Diagnose underfitting, a good fit and overfitting from two loss curves.
- Apply Dropout, weight decay, augmentation and early stopping — and know what each attacks.
- Choose a sensible `patience` and combine techniques.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print('PyTorch', torch.__version__)

## 01 · Which device you have

Everything so far ran on the CPU — on purpose. A GPU does the same maths **10–100× faster**. CPU works on any machine. Once models grow and data piles up, it becomes impractically slow.

<sub>Source: lecture slides 3–6</sub>

**`device.py`** — predict the output, then run.

In [ ]:
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu")
print(f"Using: {device}")

<details><summary>▸ Expected output</summary>

```
Using: cpu      (varies by machine)
```
</details>

## 02 · Moving the model and the data

Models move **in place**; tensors **don’t** — `.to()` returns a new tensor. One call moves every parameter of a model, nested ones included (still write `model = model.to(device)`). Data starts on the CPU, so the loop gains just two lines: move each batch before the forward pass.

<sub>Source: lecture slides 7–13</sub>

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
loader = DataLoader(TensorDataset(torch.randn(128, 10), torch.randint(0, 3, (128,))), batch_size=32, shuffle=True)
loss_fn = nn.CrossEntropyLoss()


**`move.py`** — predict the output, then run.

In [ ]:
model = nn.Sequential(
    nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
model = model.to(device)
print(next(model.parameters()).device)

for bx, by in loader:
    bx = bx.to(device)
    by = by.to(device)
    pred = model(bx)
    loss = loss_fn(pred, by)
    # ... rest of loop

<details><summary>▸ Expected output</summary>

```
cpu      (cuda:0 on a GPU machine)
```
</details>

> ⚠️ **Watch out — .to() needs an assignment**  
> For tensors, `.to(device)` returns a new tensor — the original stays where it was. You must reassign: `bx = bx.to(device)`.
>
> Writing just `bx.to(device)` without the assignment is a silent no-op — your data stays on the CPU.

🧪 *Interactive lab in the app:* **The device-mismatch simulator** — try it there to see this idea move.

## 03 · Code that runs anywhere

The gold standard: **one script, every machine** — laptop, Colab GPU, Mac — no `if` in the loop.

<sub>Source: lecture slides 14–20</sub>

**`anywhere.py`** — predict the output, then run.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
# ❶ Detect device once
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu")
# ❷ Data (stays on CPU until batched)
x = torch.randn(500, 10)
y = torch.randint(0, 3, (500,))
loader = DataLoader(TensorDataset(x, y),
                    batch_size=32, shuffle=True)
# ❸ Model → device
model = nn.Sequential(
    nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3)
).to(device)
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
# ❹ Training loop — move batches inside
for epoch in range(20):
    model.train()
    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        loss = loss_fn(model(bx), by)
        opt.zero_grad()
        loss.backward()
        opt.step()
print(f"Trained on {device}")

<details><summary>▸ Expected output</summary>

```
Trained on cpu      (varies by machine)
```
</details>

## 04 · Back to the CPU, and across machines

Coming back: NumPy and matplotlib need **CPU tensors**. Across machines: `map_location`. Use `.cpu().numpy()`, or `.detach().cpu().numpy()` if the tensor carries gradients. A GPU-saved checkpoint remembers its device; `map_location` remaps it on load.

<sub>Source: lecture slides 21–27</sub>

**`portable.py`** — predict the output, then run.

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(bx.to(device))
    pred_cpu = pred.cpu()              # back to CPU
    pred_np  = pred_cpu.numpy()        # now NumPy-safe

# Save (on GPU)
torch.save(model.state_dict(), "model.pt")
# Load (on any device, using our variable)
model.load_state_dict(
    torch.load("model.pt",
               map_location=device,
               weights_only=True))

> ⚠️ **Watch out — Don’t move the whole dataset**  
> Do not move your entire dataset to the GPU at once. GPU memory is limited (often 4–16 GB). Move each batch inside the loop, let it train, then let it be garbage-collected.
>
> A DataLoader with `pin_memory=True` speeds up CPU→GPU transfer of each batch.

## 05 · Diagnosing overfitting from two curves

A model with 99% training accuracy and 65% validation accuracy has not learned the task — it has **memorized the training data**.

<sub>Source: lecture slides 30–34</sub>

🧪 *Interactive lab in the app:* **Read the curves** — try it there to see this idea move.

🧪 *Interactive lab in the app:* **Overfitting lab — a real model, fitted live** — try it there to see this idea move.

## 06 · The regularization toolkit: Dropout and weight decay

**Regularisation**: anything that **narrows the gap** between training and validation. You’ve met several already.

<sub>Source: lecture slides 35–42</sub>

**`dropout_wd.py`** — predict the output, then run.

In [ ]:
model = nn.Sequential(
    nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(128, 10),             # no dropout on output
)
optimizer = torch.optim.Adam(
    model.parameters(), lr=1e-3,
    weight_decay=1e-4           # typical: 1e-4 to 1e-2
)

## 07 · Early stopping, and combining techniques

Best-model saving **is** early stopping. `patience` says how long to wait — 5 to 10 epochs. Too low and a noisy wobble stops you early; too high and you train past the optimum.

<sub>Source: lecture slides 43–51</sub>

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
avg_val = 0.5   # stand-in: a validation loss that has stopped improving


**`early_stop.py`** — predict the output, then run.

In [ ]:
best_val = float("inf")
patience = 5
wait = 0
for epoch in range(100):
    # ... train + validate → avg_val ...
    if avg_val < best_val:
        best_val = avg_val
        wait = 0
        torch.save(model.state_dict(), "best.pt")
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stop at epoch {epoch}")
            break
# Load the best model
model.load_state_dict(
    torch.load("best.pt", weights_only=True))

🧪 *Interactive lab in the app:* **Choosing patience** — try it there to see this idea move.

> ⚠️ **Watch out — A safety net, not a solution**  
> Early stopping is not a substitute for understanding why the model is overfitting. If a model overfits in 5 epochs, the answer is usually not “stop at epoch 4.”
>
> It is “add regularization, get more data, or simplify the model.”

## The rule — worth remembering

**Moving to GPU — six rules**

1. Define `device` once at the top. Use it everywhere — never hardcode `"cuda"`.
2. Move the model with `model.to(device)`. Move each batch inside the loop with `bx.to(device)`.
3. For tensors, `.to()` returns a new tensor — reassign or nothing moves.
4. The “Expected all tensors on the same device” error means model and data are on different devices. Fix by moving both to `device`.
5. To bring results back: `.cpu().numpy()` or `.detach().cpu().numpy()`.
6. Use `map_location=device` when loading saved models across machines.

**Overfitting & regularization — seven rules**

1. Overfitting = training loss drops, validation loss rises. Diagnose by plotting both curves.
2. Dropout (0.1–0.5) randomly silences neurons. Place after activation, never on the output layer.
3. Weight decay (1e-4 to 1e-2) penalizes large weights. One keyword in the optimizer.
4. Early stopping saves the best model and stops when validation stops improving. Use patience 5–10.
5. Data augmentation gives the model different views of the same data. Free extra data.
6. More data is the most reliable fix. Simpler model is the fastest to try.
7. Combine multiple techniques — they complement each other.

## Pause and try

Work each exercise in the empty cell below it. Open the worked answer only after you have tried.

### Exercise 01 · Try it — where is everything?

Run the device-agnostic pattern. Print `device` and `next(model.parameters()).device` to confirm everything is on the same device. On Colab, switch to a GPU runtime and run again.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Both print the same device (`cpu`, or `cuda` / `cuda:0` on a GPU runtime). Switching runtimes changes nothing in the code — only the speed.
</details>

### Exercise 02 · Ch 30 · Ex 3

Deliberately cause a device-mismatch error: model on one device, data on another. What does the message say? (Needs a GPU or a Mac.)

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!` — reproduce every variant in the simulator above.
</details>

### Exercise 03 · Ch 30 · Ex 6

Save a model on your device, then load it with `map_location="cpu"`. Does it load without errors?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Yes. `map_location` rewrites the saved device tags, so GPU-saved tensors land on the CPU.
</details>

### Exercise 04 · Ch 31 · Ex 1 & 2

Train a large model (256 → 128 → 64 → 10) on only 100 MNIST samples for 50 epochs. Plot train and val loss — does the classic overfitting pattern appear? Add `Dropout(0.5)` and retrain. Did the gap shrink?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Yes: training loss heads to zero while validation loss bottoms out and climbs — far more parameters than samples. With Dropout the training loss falls more slowly and the gap narrows, though with only 100 samples it doesn’t close.
</details>

### Exercise 05 · Try it — does weight decay close the gap?

Train a model on 100 samples with and without `weight_decay=0.01`. Plot both train and val loss curves.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Weight decay raises the training loss slightly and lowers or flattens the validation loss, shrinking the gap. Slide the weight-decay control in the overfitting lab to see the same effect on a real fit.
</details>

### Exercise 06 · Ch 31 · Ex 4 & 6

Add early stopping with patience 5 to the Exercise 1 setup — at which epoch does it stop? Then try the opposite: a model that is too small (10 → 3 → 10) on 10,000 samples. Does underfitting appear? How do you fix it?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

It stops about five epochs after the validation minimum, and `best.pt` holds the epoch before overfitting began. The tiny model shows both curves high and flat — underfitting — and the fix is more capacity: wider or more layers.
</details>

### Interview-style

(a) “How do you write PyTorch code that runs on both CPU and GPU without modification?” — three sentences, then what `map_location` does.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

Detect the device once at the top of the script and store it in a `device` variable. Move the model to it once with `model.to(device)`, and move every batch inside the training loop with `bx, by = bx.to(device), by.to(device)`. Never hardcode a device name, and bring results back with `.cpu()` before NumPy or plotting. `map_location` tells `torch.load` where to place saved tensors, remapping the device tag they were saved with.
</details>

### Interview-style

(b) “Your model gets 98% training accuracy but only 72% validation accuracy. What is happening, and what three things would you try first?” — five sentences.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

The model is overfitting: it has memorized the training data instead of learning patterns that generalize. I would confirm it by plotting training and validation loss on the same axes and looking for validation turning upward. First I’d add or increase Dropout after the activations. Second I’d add weight decay in the optimizer and data augmentation in the training transforms. Third I’d get more data or shrink the model — with early stopping as a safety net, not the fix.
</details>